In [1]:
from calculations.Composition.Composition import Composition
from calculations.Utils.Conditions import Conditions
from calculations.CompositionalModel.CompositionalModel import CompositionalModel
from calculations.EOS.PenelouxVolumeCorrection import PenelouxVolumeCorrection
from calculations.Utils.ResultsViewer import FlashResultsViewer, DLEResultsViewer, SeparatorTestResultsViewer, StandardSeparationResultsViewer
from calculations.Utils.CompositionLoader import CompositionExcelLoader

# KRSNLN

In [ ]:
excel_loader = CompositionExcelLoader(r'C:\Users\user\Desktop\PVT_TSU\diss\krsnln.xlsx')
krsnln_dict = excel_loader.load(header=True, sheet='to_model')

In [ ]:
krsnln_dict

In [ ]:
krsnln_base_composition = Composition(zi = krsnln_dict)

In [ ]:
krsnln_model = CompositionalModel(krsnln_base_composition, eos= 'PREOS')

In [ ]:
conds = Conditions(5,50)
krsnln_model.flash(conditions=conds)

In [ ]:
krsnln_model.show_flashes

In [ ]:
krsnln_model.experiments.SEPARATORTEST.calculate_3stages([0.7, 0.5, 0.1], [20, 20, 20])

In [ ]:
krsnln_model.experiments.SEPARATORTEST.result

In [ ]:
from calculations.Utils.ResultsViewer import SeparatorTestResultsViewer

sep_test_view = SeparatorTestResultsViewer()
sep_test_view.view(krsnln_model.experiments.SEPARATORTEST.result)

# PRRZLM №200839

In [2]:
excel_loader = CompositionExcelLoader(r'C:\Users\user\Desktop\PVT_TSU\diss\prrzlm.xlsx')
przlm_dict = excel_loader.load(header=True, sheet='to_model')

### Фактические данные

In [79]:
p_sat = 120.47


## Создаем базовую модель и сравниваемся с PVTSIM

In [3]:
przlm_comp = Composition(przlm_dict)

In [4]:
przlm_model = CompositionalModel(przlm_comp)

In [5]:
conds = Conditions(5,50)

In [6]:
przlm_model.flash(conds)

In [7]:
przlm_model.saturation_pressure(60)

SATURATION PRESSURE CALCULATION
=====
CALCULATED SATURATION PRESSURE: 85.35510856474869 bar
TEMPERATURE: 60 C


### Проверяем сепаратор тест

In [8]:
przlm_model.experiments.SEPARATORTEST.calculate_3stages([0.7, 0.5, 0.1], [20, 20, 20])

SeparatorTestResults(first_stage_pressure=0.7, first_stage_temperature=293.14, first_stage_fv=0.2861830815188178, first_stage_fl=0.7138169184811822, first_stage_vapour_composition={'CO2': 0.0029932677287709797, 'N2': 0.04090762837866887, 'C1': 0.8455156478353413, 'C2': 0.07249921176560595, 'C3': 0.025001469679088433, 'iC4': 0.0030182128152255656, 'nC4': 0.006183186143875353, 'iC5': 0.0013864810540705954, 'nC5': 0.0009647890905620216, 'C6': 0.0006899637678559609, 'C7': 0.00045716462315847087, 'C8': 0.0002677062185067711, 'C9': 7.627626308400961e-05, 'C10': 2.361043749410221e-05, 'C11': 7.913562093545391e-06, 'C12': 2.910790380170107e-06, 'C13': 1.0552135609902687e-06, 'C14': 3.467166589485652e-07, 'C15': 1.0748351262488413e-07, 'C16': 3.5156093032476375e-08, 'C17': 9.534763243779457e-09, 'C18': 3.5222215671723997e-09, 'C19': 1.4433624424841594e-09, 'C20': 5.30103980894262e-10, 'C21': 1.5696826254973423e-10, 'C22': 6.131901202427214e-11, 'C23': 1.8261377590468838e-11, 'C24': 6.2510250944

## Начинаем играть корреляциями

In [86]:
prr_base_comp = Composition(przlm_dict, c6_plus_correlations= {'critical_temperature': 'pedersen',
                                                    'critical_pressure': 'rizari_daubert',
                                                    'acentric_factor': 'rizari_al_sahhaf',
                                                    'critical_volume': 'hall_yarborough',
                                                    'k_watson': 'k_watson',
                                                    'shift_parameter': 'jhaveri_youngren'})


In [62]:
prr_kesler_comp = Composition(przlm_dict, c6_plus_correlations= {'critical_temperature': 'pedersen',
                                                    'critical_pressure': 'pedersen',
                                                    'acentric_factor': 'rizari_al_sahhaf',
                                                    'critical_volume': 'hall_yarborough',
                                                    'k_watson': 'k_watson',
                                                    'shift_parameter': 'jhaveri_youngren'})
prr_kesler_comp.COMPOSITION_PROPERTIES.describe()

,molar_mass,gamma,Tb,critical_pressure,critical_temperature,acentric_factor,shift_parameter,critical_volume,c5_plus_flag
count,40.000000,40.000000,40.000000,40.000000,40.000000,40.000000,40.000000,40.000000,40.000000
mean,235.324025,0.777948,513.599500,2.317859,646.040657,0.707354,0.108961,15.502312,0.775000
std,151.968692,0.157837,200.195077,1.312239,201.921390,0.429286,0.145697,10.282038,0.422902
min,16.043000,0.330000,77.390000,1.311351,126.190000,0.013000,-0.192700,0.090100,0.000000
25%,93.544500,0.717750,358.050000,1.407845,527.029460,0.293227,-0.020308,6.078818,1.000000
50%,229.500000,0.847000,564.650000,1.675149,697.472980,0.724813,0.150584,14.792799,1.000000
75%,362.750000,0.893000,683.900000,3.077022,805.346890,1.075208,0.236398,24.015258,1.000000
max,500.000000,0.922000,766.150000,7.387000,898.796296,1.403862,0.291305,33.864255,1.000000


In [74]:
model_base = CompositionalModel(prr_base_comp)
model_kesler = CompositionalModel(prr_kesler_comp)

In [77]:
model_kesler.experiments.SEPARATORTEST_MOD.calculate(150, 60, [0.76,0.3,0.101], [55, 58.3, 20])

GVOL BY STAGES STC: [0, 0, 7417.573431956575, 378.4647704996596, 135.6766868804984, 0]
GVOL BY STAGES ACC STC: [0, 0, 7417.573431956575, 7796.038202456235, 7931.714889336733, 7931.714889336733]


DLEResults(index=['150_333.14', '10.110507365444207_333.14', '0.76_331.44', '0.3_328.14', '0.101_293.14', 'STC_0.101325_293.14'], pressure_arr=[150, 10.110507365444207, 0.76, 0.3, 0.101, 0.101325], temperature_arr=[333.14, 333.14, 331.44, 328.14, 293.14, 293.14], liquid_volume_arr=[193.23608340919742, 205.1059513908803, 188.85077045405106, 272.1090088192207, 273.91756533559925, 278.5352288129547], gas_volume_arr=[193.23608340919742, 0.0003731513992303443, 1104.4253404904598, 206.38992248193895, 202.00671754930397, 278.5352288129547], liquid_density_arr=[0.8862718200375644, 0.8349845183295471, 0.872752924722444, 0.8762002319020133, 0.8900800380634968, 0.8827606653095227], gas_density_arr=[0.8862718200375644, 0.07656444598562377, 0.005831536442797236, 0.0027433143928842015, 0.0010703278967069284, 0.8827606653095227], fl_arr=[None, 0.9999984535689777, 0.6896578307112406, 0.9770400612775818, 0.9915756165954561, None], fv_arr=[None, 1.5464310223478606e-06, 0.31034216928875935, 0.02295993872

In [85]:
model_base.saturation_pressure(58)

SATURATION PRESSURE CALCULATION
=====
CALCULATED SATURATION PRESSURE: 87.05150362482529 bar
TEMPERATURE: 58 C


In [78]:
dleresview = DLEResultsViewer()
dleresview.view(model_kesler.experiments.SEPARATORTEST_MOD.result)

,Index,Pressure,Temperature,Fl,Fv,Liquid Z,Gas Z,Liquid volume,Gas volume,Liquid density,Gas density,Bo,Rs,Liquid viscosity,Gas viscosity
0,150_333.14,150.000000,333.14,NaN,NaN,10.470121,10.470103,193.236083,193.236083,0.886272,0.886272,1.038334,42.620226,None,None
1,10.110507365444207_333.14,10.110507,333.14,0.999998,0.000002,0.749073,0.881251,205.105951,0.000373,0.834985,0.076564,1.102115,42.620226,None,None
2,0.76_331.44,0.760000,331.44,0.689658,0.310342,0.075560,0.981981,188.850770,1104.425340,0.872753,0.005832,1.014768,2.762684,None,None
3,0.3_328.14,0.300000,328.14,0.977040,0.022960,0.030640,0.988960,272.109009,206.389922,0.876200,0.002743,1.008381,0.729044,None,None
4,0.101_293.14,0.101000,293.14,0.991576,0.008424,0.011454,0.994198,273.917565,202.006718,0.890080,0.001070,0.991777,0.000000,None,None
5,STC_0.101325_293.14,0.101325,293.14,NaN,NaN,0.011586,0.994174,278.535229,278.535229,0.882761,0.882761,1.000000,0.000000,None,None
